In [24]:
import re
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, MinMaxScaler # MinMaxScaler eklendi
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier

Burada kullandıklarımızın görevi:

pandas → veri setini okumak

numpy → sayısal işlemler

re → metin içinde link, kelime, sembol aramak

TfidfVectorizer → metni sayısal hale getirmek

StandardScaler → sayısal özellikleri ölçeklemek

VotingClassifier → birden fazla modeli birleştirmek

In [2]:
from google.colab import files
uploaded = files.upload()

Saving spam.csv to spam.csv


Bu kod çalışınca Colab sana dosya seçtirir.
Buradan spam.csv dosyanı seçersin.

In [3]:
df = pd.read_csv("spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


Bu veri setinde genelde şu sütunlar olur:

v1 → etiket (ham / spam)

v2 → mesaj metni

bazen ek boş sütunlar da gelir

encoding="latin-1" kullanmamızın sebebi bu veri setinde bazı özel karakterlerin hata vermesini önlemektir.

In [4]:
df = df.iloc[:, :2]
df.columns = ["label", "message"]

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Burada ilk iki sütunu aldık. Sütun isimlerini daha anlaşılır yaptık:

label / message

Artık veri setimiz daha temiz.

In [5]:
df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


Makine öğrenmesi modelleri metin etiketlerle değil, sayılarla daha rahat çalışır.

Burada:
ham = 0
spam = 1  olarak çevirdik.

Yani model artık:

0 → normal mesaj

1 → spam mesaj               

şeklinde öğrenecek.

In [6]:
print("Veri boyutu:", df.shape)
print("\nSınıf dağılımı:")
print(df["label"].value_counts())

Veri boyutu: (5572, 2)

Sınıf dağılımı:
label
0    4825
1     747
Name: count, dtype: int64


Bu adım önemli çünkü:

kaç satır var görürüz,

spam ve ham sayısı dengeli mi görürüz.

Spam veri setlerinde genelde ham sayısı daha fazladır.
Bu da sınıf dengesizliği açısından yorum yapmamızı sağlar.

**----------------------------------------------------------------------------**

Şimdi hibrit modelin önemli kısmına geliyoruz.
Sadece TF-IDF kullanmayacağız. Mesajdan ayrıca bazı ek sinyaller çıkaracağız.

In [7]:
def extract_handcrafted_features(text_series):
    features = []

    for text in text_series:
        text = str(text)

        msg_len = len(text)
        word_count = len(text.split())
        digit_count = sum(c.isdigit() for c in text)
        exclamation_count = text.count("!")
        question_count = text.count("?")
        uppercase_count = sum(c.isupper() for c in text)
        uppercase_ratio = uppercase_count / max(len(text), 1)
        url_count = len(re.findall(r"http[s]?://|www\.", text))
        money_symbol_count = text.count("$") + text.count("£")
        free_word_count = len(re.findall(r"\bfree\b", text.lower()))
        win_word_count = len(re.findall(r"\bwin\b|\bwinner\b|\bwon\b", text.lower()))

        features.append([
            msg_len,
            word_count,
            digit_count,
            exclamation_count,
            question_count,
            uppercase_count,
            uppercase_ratio,
            url_count,
            money_symbol_count,
            free_word_count,
            win_word_count
        ])

    return np.array(features, dtype=float)

Bu fonksiyon her mesajdan şu özellikleri çıkarıyor:

msg_len → mesaj uzunluğu

word_count → kelime sayısı

digit_count → rakam sayısı

exclamation_count → ! sayısı

question_count → ? sayısı

uppercase_count → büyük harf sayısı

uppercase_ratio → büyük harf oranı

url_count → link var mı

money_symbol_count → $ veya £ var mı

free_word_count → free kelimesi kaç kez geçmiş

win_word_count → win, winner, won kaç kez geçmiş


**Neden önemli?**

Çünkü spam mesajlar çoğu zaman şöyle olur:

çok büyük harf,

çok ünlem,

link içerme,

para sembolü,

free, win gibi kelimeler

Yani bu özellikler spam’i yakalamada yardımcı olur.

**--------------------------------------------------------------------------------**

Şimdi giriş ve hedef değişkenleri ayırıyoruz.

In [9]:
X_text = df["message"]
y = df["label"]

X_text → modelin göreceği mesaj metinleri

y → gerçek etiketler (0 veya 1)

In [10]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Burada:

%80 eğitim

%20 test

olarak ayırdık.

Parametreler ne işe yarıyor?

test_size=0.2 → verinin %20’si test

random_state=42 → her çalıştırmada aynı bölünme olsun.

stratify=y → sınıf dağılımı train ve testte benzer kalsın.

Bu çok önemli. Çünkü spam ve ham oranı bozulmasın istiyoruz.

**--------------------------------------------------------------------------------**

Şimdi metinleri sayısal özelliklere dönüştürüyoruz.

In [11]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=5000
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

**TfidfVectorizer ne yapar?**

Kelimeleri sayısal vektörlere çevirir.

Örneğin model artık şu tür şeyleri görebilir:

“free”

“call now”

“claim prize”

“urgent reply”

Parametreler

lowercase=True → metni küçük harfe çevirir

stop_words="english" → gereksiz İngilizce bağlaçları çıkarır

ngram_range=(1, 2) → hem tek kelime hem iki kelimelik yapılar kullanılır

örn: free

örn: free entry

max_features=5000 → en fazla 5000 özellik kullan

Bu spam tespiti için gayet iyi bir başlangıçtır.

**--------------------------------------------------------------------------------**

Biraz önce yazdığımız fonksiyonla mesajlardan ek özellikleri çıkarıyoruz.

In [12]:
X_train_num = extract_handcrafted_features(X_train_text)
X_test_num = extract_handcrafted_features(X_test_text)

print("Sayısal özellik boyutu:", X_train_num.shape)

Sayısal özellik boyutu: (4457, 11)


Açıklama

Bu adımda artık her mesaj için 11 tane ek özellik üretmiş olduk.

Örneğin bir mesaj şu sinyalleri taşıyabilir:

uzunluk, link içeriyor mu?, para sembolü var mı?, fazla ünlem var mı?

Bunlar TF-IDF dışında ekstra bilgi sağlar.

**--------------------------------------------------------------------------------**

Bu sayısal özelliklerin değer aralıkları farklı olduğu için ölçekliyoruz.

In [29]:
scaler = MinMaxScaler() # StandardScaler yerine MinMaxScaler kullanıldı

X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)

Örneğin:

Mesaj uzunluğu 150 olabilir, ünlem sayısı 2 olabilir, büyük harf oranı 0.13 olabilir.

Bu değerler farklı ölçeklerde olduğu için StandardScaler ile normalize ediyoruz.
Bu, bazı modellerin daha dengeli çalışmasını sağlar.

**--------------------------------------------------------------------------------**

TF-IDF çıktısı sparse matristir. Sayısal özellikleri de onunla birleştirebilmek için sparse formata çeviriyoruz.

In [30]:
X_train_num_sparse = csr_matrix(X_train_num_scaled)
X_test_num_sparse = csr_matrix(X_test_num_scaled)

Bu teknik bir adımdır.
Ama gerekli çünkü hstack ile TF-IDF ve sayısal veriyi birleştireceğiz.

In [31]:
X_train_hybrid = hstack([X_train_tfidf, X_train_num_sparse])
X_test_hybrid = hstack([X_test_tfidf, X_test_num_sparse])

print("Train hybrid shape:", X_train_hybrid.shape)
print("Test hybrid shape:", X_test_hybrid.shape)

Train hybrid shape: (4457, 5011)
Test hybrid shape: (1115, 5011)


Burada iki farklı bilgi kaynağını birleştirdik:

**TF-IDF özellikleri**

Mesajın içeriğini temsil eder
örnek:

free

call

prize

urgent

**Elle çıkarılmış sayısal özellikler**

Mesajın yapısını temsil eder
örnek:

kaç ünlem var

link var mı

para sembolü var mı

büyük harf oranı ne

İşte bu yüzden buna hibrit özellik yapısı diyoruz.

In [33]:
y_pred = hybrid_model.predict(X_test_hybrid)

In [20]:
log_reg = LogisticRegression(max_iter=2000, random_state=42)
nb = ComplementNB()

svm = CalibratedClassifierCV(
    estimator=LinearSVC(random_state=42),
    cv=3
)

hybrid_model = VotingClassifier(
    estimators=[
        ("lr", log_reg),
        ("nb", nb),
        ("svm", svm)
    ],
    voting="soft"
)

Burada 3 model kullanıyoruz:

1. Logistic Regression

Metin sınıflandırmada güçlü ve dengeli bir modeldir.

2. Multinomial Naive Bayes

Spam tespitinde klasik ve hızlıdır.

3. Linear SVM

Özellikle metin verisinde çoğu zaman çok başarılı olur.

**Neden CalibratedClassifierCV kullandık?**

LinearSVC doğrudan olasılık vermez.
Ama VotingClassifier içinde soft voting kullanmak için olasılık gerekir.

Bu yüzden SVM modelini kalibre ediyoruz.

**soft voting ne demek?**

Her model sadece “spam” ya da “ham” demekle kalmaz, aynı zamanda bir olasılık tahmini yapar.
Sonra bu olasılıklar birleştirilir.

Bu, çoğu zaman tek modelden daha iyi sonuç verebilir.

In [32]:
hybrid_model.fit(X_train_hybrid, y_train)

VotingClassifier(estimators=[('lr',
                              LogisticRegression(max_iter=2000,
                                                 random_state=42)),
                             ('nb', ComplementNB()),
                             ('svm',
                              CalibratedClassifierCV(cv=3,
                                                     estimator=LinearSVC(random_state=42)))],
                 voting='soft')

Bu aşamada model:

TF-IDF özelliklerinden, ek sayısal özelliklerden ve 3 farklı algoritmanın ortak kararından öğrenmiş olur.

In [34]:
y_pred = hybrid_model.predict(X_test_hybrid)

Bu satır her test mesajı için:

0 → ham

1 → spam

tahmini üretir.

In [35]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9874439461883409

Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.92      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115

Confusion Matrix:

[[964   2]
 [ 12 137]]


Model %98.7 doğruluk oranı elde ederek oldukça yüksek bir performans göstermiştir. Özellikle normal mesajları tespit etmede neredeyse hatasız çalışmaktadır. Spam mesajlar için de yüksek precision ve recall değerleri elde edilmiştir. Confusion matrix incelendiğinde yalnızca az sayıda spam mesajın kaçırıldığı görülmektedir. Bu sonuçlar hibrit yaklaşımın (TF-IDF + ek metin özellikleri + ensemble model) spam tespiti için oldukça etkili olduğunu göstermektedir.

In [36]:
train_pred = hybrid_model.predict(X_train_hybrid)

print("Training Accuracy:", accuracy_score(y_train, train_pred))

Training Accuracy: 0.9959614090195199


In [37]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    hybrid_model,
    X_train_hybrid,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("CV scores:", scores)
print("CV mean:", scores.mean())

CV scores: [0.99103139 0.98766816 0.98989899 0.98540965 0.98989899]
CV mean: 0.9887814366887611


Model yüksek doğruluk elde etmiş olsa da, değerlendirme ayrı bir test veri seti üzerinde yapıldığı için aşırı öğrenme kesin olarak söylenemez. Ayrıca spam mesajlar belirgin kelime ve yazım kalıpları içerdiğinden makine öğrenmesi modelleri bu veri setinde yüksek performans elde edebilmektedir.